In [1]:
import glob
import pandas as pd
from tqdm.auto import tqdm

/Users/soeren.becker/micromamba/envs/env_pertpy/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### DEs as computed on pooled PBS wells

In [2]:
deg_raw = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_unfiltered.csv", index_col=0)

In [3]:
raw_celltypes = deg_raw.celltype.unique()
raw_celltypes

array(['B_Intermediate', 'B_Naive', 'B', 'CD4_Memory', 'CD4_Naive', 'CD4',
       'CD8_Memory', 'CD8_Naive', 'CD8', 'CD14_Mono', 'CD16_Mono',
       'CD56-bright_NK', 'CD56-dim_NK', 'cDC', 'HSPC', 'ILC', 'MAIT',
       'Mono', 'NK', 'NKT', 'other', 'pDC', 'Plasmablast', 'Treg'],
      dtype=object)

### load list genes to be filtered out

In [6]:
blacklisted_genes = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/notebooks/01_overview/questionable_genes.csv").gene.tolist()

### DEs as computed vs individual PBS wells

In [5]:
paths = glob.glob("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_*_separate_pbs_wells.csv")
len(paths)

24

In [6]:
paths

['/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_B_Intermediate_Memory_separate_pbs_wells.csv',
 '/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_NKT_separate_pbs_wells.csv',
 '/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_CD8_Memory_separate_pbs_wells.csv',
 '/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_CD56-dim_NK_separate_pbs_wells.csv',
 '/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_CD16_Mono_separate_pbs_wells.csv',
 '/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_CD56-bright_NK_separate_pbs_wells.csv',
 '/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_CD8_Naive_separate_pbs_wells.csv',
 '/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_pDC_separate_pbs_wells.csv',
 '/Users/soeren.becker/r

In [7]:
processed_DEGs = []
for path in tqdm(paths):
    print(path)
    df = pd.read_csv(path, index_col=0)
    
    assert len(df.celltype.unique())
    celltype = df.celltype.unique()[0]
    if celltype == "B_Intermediate_Memory":
        celltype = "B_Intermediate"

    if not celltype in raw_celltypes:
        print(celltype, path)
        continue
    
    df_well_counts = df.loc[
        (
            (~df.variable.isin(blacklisted_genes))
            & (df.log_fc.abs() > 0.25) 
            & (df.adj_p_value < 0.1)
        )
    ].groupby(["cytokine", "variable"])["pbs_well"].nunique().to_frame().reset_index()
    
    deg_w_DE_count = pd.merge(deg_raw.loc[deg_raw.celltype == celltype], df_well_counts, how="outer")
    
    deg_w_DE_count.loc[:, "pbs_well"] = deg_w_DE_count["pbs_well"].fillna(0)
    processed_DEGs.append(deg_w_DE_count)
DEGs_with_DE_count = pd.concat(processed_DEGs, axis=0)

  0%|                                                                                                                                                                                | 0/24 [00:00<?, ?it/s]

/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_B_Intermediate_Memory_separate_pbs_wells.csv


  4%|███████                                                                                                                                                                 | 1/24 [00:07<02:47,  7.27s/it]

/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_NKT_separate_pbs_wells.csv


  8%|██████████████                                                                                                                                                          | 2/24 [00:14<02:37,  7.14s/it]

...

 92%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████              | 22/24 [02:25<00:15,  7.91s/it]

/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_NK_separate_pbs_wells.csv


 96%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████       | 23/24 [02:29<00:06,  6.86s/it]

/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_separate_pbs_wells/DEGs_Mono_separate_pbs_wells.csv


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 24/24 [02:37<00:00,  6.56s/it]


In [8]:
for celltype in deg_raw.celltype.unique():

    print(celltype)
    print(deg_raw.loc[deg_raw.celltype == celltype].shape)
    print(DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == celltype].shape)
    print(DEGs_with_DE_count.loc[(DEGs_with_DE_count.celltype == celltype) & (DEGs_with_DE_count.pbs_well > 3)].shape)
    
    print("---")

B_Intermediate
(798218, 18)
(798218, 19)
(34626, 19)
---
B_Naive
(644821, 18)
(644821, 19)
(39279, 19)
---
B
(712867, 18)
(712867, 19)
(46861, 19)
---
CD4_Memory
(701282, 18)
(701282, 19)
(46433, 19)
---
CD4_Naive
(667343, 18)
(667343, 19)
(43107, 19)
---
CD4
(685117, 18)
(685117, 19)
(43234, 19)
---
CD8_Memory
(686579, 18)
(686579, 19)
(43227, 19)
---
CD8_Naive
(713680, 18)
(713680, 19)
(47678, 19)
---
CD8
(706214, 18)
(706214, 19)
(44677, 19)
---
CD14_Mono
(842289, 18)
(842289, 19)
(99251, 19)
---
CD16_Mono
(815610, 18)
(815610, 19)
(57466, 19)
---
CD56-bright_NK
(813763, 18)
(813763, 19)
(19249, 19)
---
CD56-dim_NK
(687656, 18)
(687656, 19)
(42802, 19)
---
cDC
(1069265, 18)
(1069265, 19)
(47403, 19)
---
HSPC
(1029180, 18)
(1029180, 19)
(2328, 19)
---
ILC
(275754, 18)
(275754, 19)
(7, 19)
---
MAIT
(721347, 18)
(721347, 19)
(23931, 19)
---
Mono
(840196, 18)
(840196, 19)
(98049, 19)
---
NK
(713844, 18)
(713844, 19)
(49621, 19)
---
NKT
(697433, 18)
(697433, 19)
(12187, 19)
---
other
(46

In [9]:
DEGs_with_DE_count.loc[
    (
        (~DEGs_with_DE_count.variable.isin(blacklisted_genes))
        & (DEGs_with_DE_count.adj_p_value < 0.05) 
        & (DEGs_with_DE_count.log_fc.abs() > 0.25) 
        & (DEGs_with_DE_count.pbs_well > 3)
    )
].groupby(["celltype"]).variable.count().sort_values()

celltype
ILC                   6
Plasmablast          39
other               135
pDC                 970
HSPC               2094
NKT               11731
CD56-bright_NK    19071
Treg              21437
MAIT              23254
B_Intermediate    33233
B_Naive           37240
CD4_Naive         40087
CD8_Memory        40371
CD4               40395
CD56-dim_NK       40418
CD8               41429
CD4_Memory        43272
B                 44000
CD8_Naive         44190
cDC               46604
NK                46838
CD16_Mono         55258
Mono              92257
CD14_Mono         93288
Name: variable, dtype: int64

### Some more formatting

In [10]:
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "other", "celltype"] = "Granulocyte"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD56-bright_NK", "celltype"] = "NK_CD56hi"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD56-dim_NK", "celltype"] = "NK_CD56low"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "B_Naive", "celltype"] = "Naive_B_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "B_Intermediate", "celltype"] = "Intermediate_B_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD4", "celltype"] = "CD4_T_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD4_Naive", "celltype"] = "CD4_Naive_T_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD4_Memory", "celltype"] = "CD4_Memory_T_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD8", "celltype"] = "CD8_T_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD8_Naive", "celltype"] = "CD8_Naive_T_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "CD8_Memory", "celltype"] = "CD8_Memory_T_cell"
DEGs_with_DE_count.loc[DEGs_with_DE_count.celltype == "B", "celltype"] = "B_cell"

DEGs_with_DE_count.rename({"variable": "gene"}, axis=1, inplace=True)

In [11]:
sorted(DEGs_with_DE_count.celltype.unique())

['B_cell',
 'CD14_Mono',
 'CD16_Mono',
 'CD4_Memory_T_cell',
 'CD4_Naive_T_cell',
 'CD4_T_cell',
 'CD8_Memory_T_cell',
 'CD8_Naive_T_cell',
 'CD8_T_cell',
 'Granulocyte',
 'HSPC',
 'ILC',
 'Intermediate_B_cell',
 'MAIT',
 'Mono',
 'NK',
 'NKT',
 'NK_CD56hi',
 'NK_CD56low',
 'Naive_B_cell',
 'Plasmablast',
 'Treg',
 'cDC',
 'pDC']

In [12]:
DEGs_with_DE_count.to_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_sanity_check.csv")

In [2]:
DEGs_with_DE_count = pd.read_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_sanity_check.csv")

In [5]:
DEGs_with_DE_count = DEGs_with_DE_count.set_index("Unnamed: 0")

In [7]:
DEGs_with_DE_count.shape

(17180340, 19)

In [8]:
DEGs_with_DE_count = DEGs_with_DE_count.loc[~DEGs_with_DE_count.gene.isin(blacklisted_genes)]

In [9]:
DEGs_with_DE_count.shape

(16852170, 19)

In [10]:
DEGs_with_DE_count.rename({"pbs_well": "num_DE_pbs_wells"}, axis=1, inplace=True)

In [13]:
DEGs_with_DE_count.to_csv("/Users/soeren.becker/repos/HumanCytokineDict/data/DEGs/DEGs_all_with_pbs_DE_count_with_blacklisted_genes_removed.csv")